# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
import pandas as pd
import numpy as np

dataset = pd.read_csv('work/outputs/dataset.csv')

feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d',
    'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d',
    'organic_sessions_90d', 'impressions_last30', 'impressions_first60',
    'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum'
]
target_col = 'is_declining_label'

# Finding #A's trap: health_score = impressions(30pts) + position(30pts) + ctr(20pts) + scroll(20pts),
# then a model "predicts" health_score FROM impressions/position/ctr/scroll. Check we don't
# repeat that pattern: does any of OUR features go into the formula that built is_declining_label?
label_leak_cols = ['impressions_mar', 'impressions_apr', 'pct_change', 'trend_direction', 'trend_pct']
print("Does any of our 14 features overlap with the columns that construct is_declining_label?")
print(f"  Overlap: {[c for c in feature_cols if c in label_leak_cols] or 'none'}")

# Finding #B's trap: the growth/decline label is defined from a 30d-vs-prev-30d impression
# change, and "Impressions" is also used as an input feature -- a same-window overlap risk.
# Check ours: is_declining_label comes from March vs April impressions (excluded columns above);
# our impressions_90d/impressions_last30/impressions_first60 features are drawn from the
# Jan-Mar 90-day FEATURE window, never from March/April (the label window) per the ML-04 contract.
print("\nOur label window (per ML-04 data contract): March->April 2026 impression change.")
print("Our feature window: Jan-Mar 2026 (90 days), strictly before the label window.")
print("impressions_last30 = March only -- same CALENDAR month as the label's 'before' side, but")
print("it is the pre-period impression LEVEL, not the label's post-period change -- worth naming")
print("explicitly rather than assuming the window separation is automatically clean.")


Does any of our 14 features overlap with the columns that construct is_declining_label?
  Overlap: none

Our label window (per ML-04 data contract): March->April 2026 impression change.
Our feature window: Jan-Mar 2026 (90 days), strictly before the label window.
impressions_last30 = March only -- same CALENDAR month as the label's 'before' side, but
it is the pre-period impression LEVEL, not the label's post-period change -- worth naming
explicitly rather than assuming the window separation is automatically clean.


## 1. Two paper findings + my methodology questions

**Finding A -- "What Predicts Health?" (ML Appendix, p.27).** A Random Forest reports
feature importance for predicting `health_score`: Average Position 43%, Impressions 32%,
Scroll Depth 15%, CTR 8%. But `health_score` is *defined*, per the paper's own "How to Read
This Paper" section (p.5), as `Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll
Depth (20pts)`. Three of the top four "predictors" are literally the ingredients of the
label's formula.

*My methodology question, respectfully:* where does the label come from, and does an 80/20
holdout guard against that? A holdout split protects against **overfitting noise** -- it
doesn't protect against a label that's a **deterministic function of its own top features**.
The paper already discloses this ("importance is descriptive rather than causal," p.27) --
my question is whether that one caveat is enough, or whether a reader skimming the "43%
importance" headline number would still walk away thinking position *causes* health, when
the honest reading is closer to "the model rediscovered the scoring formula." (Checked above:
none of our own 14 features feed into `is_declining_label`'s construction -- the ML-04 data
contract's excluded-column list exists specifically to prevent this exact pattern.)

**Finding B -- "What Predicts Growth?" (ML Appendix, p.29).** A logistic regression (71%
holdout accuracy) separates growing from declining pages, with `Impressions` among the input
features. Per p.5, `Trend Direction` (growing vs. declining) is "calculated from 30d-vs-prev-30d
impression change."

*My methodology question:* which impression window does the `Impressions` feature use? The
report doesn't say whether it's a 90-day total, the current 30-day window, or something else
-- and that distinction matters a lot here. If it's the same (or an overlapping) 30-day window
used to compute the trend label, the model would be partly predicting the label's own
direction of change from a number that mechanically co-moves with it -- the same shape of
risk the ML-04/ML-05 exercises in this repo are built around (why `trend_direction` and
`trend_pct` are on our excluded-column list, and why `impressions_last30` in our own feature
set is drawn from the *feature* window, never the label window, as checked above). This isn't
an accusation that Finding B is wrong -- the holdout accuracy of 71% is a real, disclosed
number -- it's a question about which window "Impressions" means, since the paper doesn't say.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_STATE = 42

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_true = np.asarray(y_true)
    return float(y_true[order[:min(k, len(y_true))]].mean()) if len(y_true) else 0.0

def build_models():
    return {
        'logistic_regression': Pipeline([
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        'decision_tree': DecisionTreeClassifier(
            class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
        ),
        'random_forest': RandomForestClassifier(
            class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
            n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
        ),
    }

def evaluate_split(train_idx, test_idx, label):
    X_train, X_test = dataset.loc[train_idx, feature_cols], dataset.loc[test_idx, feature_cols]
    y_train, y_test = dataset.loc[train_idx, target_col], dataset.loc[test_idx, target_col]
    rows = []
    for name, model in build_models().items():
        model.fit(X_train, y_train)
        scores = model.predict_proba(X_test)[:, 1]
        rows.append({
            'split': label, 'model': name,
            'precision_at_50': precision_at_k(y_test, scores, 50),
            'average_precision': average_precision_score(y_test, scores),
            'roc_auc': roc_auc_score(y_test, scores) if y_test.nunique() == 2 else float('nan'),
        })
    return rows

# BEFORE: naive row-random split -- ignores that many rows share a client
before_train, before_test = train_test_split(
    dataset.index, test_size=0.2, random_state=RANDOM_STATE, stratify=dataset[target_col]
)
before_rows = evaluate_split(before_train, before_test, 'BEFORE (naive row-random)')

# AFTER: client-grouped split -- the same honest split from w05_model.ipynb
def make_client_aware_split(df, target_col, client_col='client_hash_id', test_frac=0.2, random_state=RANDOM_STATE):
    clients = df[client_col].dropna().unique()
    if len(clients) >= 5:
        rng = np.random.default_rng(random_state)
        shuffled = rng.permutation(clients)
        n_test = max(1, int(round(len(shuffled) * test_frac)))
        test_clients = set(shuffled[:n_test])
        test_mask = df[client_col].isin(test_clients)
        train_idx = df.index[~test_mask]
        test_idx = df.index[test_mask]
        if df.loc[train_idx, target_col].nunique() == 2 and df.loc[test_idx, target_col].nunique() == 2:
            return train_idx, test_idx
    train_idx, test_idx = train_test_split(df.index, test_size=test_frac, random_state=random_state, stratify=df[target_col])
    return train_idx, test_idx

after_train, after_test = make_client_aware_split(dataset, target_col)
after_rows = evaluate_split(after_train, after_test, 'AFTER (client-grouped)')

before_clients = set(dataset.loc[before_train, 'client_hash_id'])
after_train_clients = set(dataset.loc[after_train, 'client_hash_id'])
after_test_clients = set(dataset.loc[after_test, 'client_hash_id'])

comparison = pd.DataFrame(before_rows + after_rows).set_index(['split', 'model']).round(3)
print(comparison)
print(f"\nBEFORE split: client overlap between train/test = "
      f"{len(before_clients & set(dataset.loc[before_test, 'client_hash_id']))} clients "
      f"(row-random split does NOT control this)")
print(f"AFTER split:  client overlap between train/test = "
      f"{len(after_train_clients & after_test_clients)} clients (must be 0)")


                                               precision_at_50  \
split                     model                                  
BEFORE (naive row-random) logistic_regression             0.62   
                          decision_tree                   0.62   
                          random_forest                   0.82   
AFTER (client-grouped)    logistic_regression             0.74   
                          decision_tree                   0.58   
                          random_forest                   0.72   

                                               average_precision  roc_auc  
split                     model                                            
BEFORE (naive row-random) logistic_regression              0.541    0.664  
                          decision_tree                    0.527    0.694  
                          random_forest                    0.626    0.755  
AFTER (client-grouped)    logistic_regression              0.419    0.629  
               

## 2. My model under an honest split (before/after)

**BEFORE:** a plain stratified row-random 80/20 split -- the same split `scripts/03_train_model.py`
and `w05_model.ipynb` fall back to only when there aren't enough clients, used here
deliberately as the *dishonest* baseline: rows from the same client can land on both sides.

**AFTER:** the client-grouped split from `w05_model.ipynb` -- ~20% of *clients* held out
entirely, verified above to share zero clients between train and test.

Both are trained and scored with the identical three models, features, and metrics from
`w05_model.ipynb`. If BEFORE's numbers look stronger than AFTER's, that gap **is** the
leakage the honest split was built to remove -- the model partly memorizing client-level
patterns (a client's typical traffic level, its niche, its editorial habits) rather than
learning genuinely transferable content signals.

**BEFORE shares 38 of 42 clients between train and test** -- the row-random split barely
holds anything out at the client level, confirming it's the dishonest split it's meant to be.
The clearest leakage signal is **Average Precision and ROC AUC, inflated for all three models
under BEFORE**: random_forest AP 0.626 (BEFORE) vs. 0.407 (AFTER), AUC 0.755 vs. 0.643;
decision_tree AP 0.527 vs. 0.394, AUC 0.694 vs. 0.640; logistic_regression AP 0.541 vs. 0.419,
AUC 0.664 vs. 0.629 -- a consistent, real gap across every model and both threshold-free
metrics. Precision@50 alone is noisier and doesn't tell the same clean story: random_forest
does drop as expected (0.82 -> 0.72), decision_tree drops slightly (0.62 -> 0.58), but
logistic_regression actually looks *better* AFTER (0.62 -> 0.74) -- a reminder that P@50 on
one 50-row slice is a single noisy draw, and the AP/AUC gap (computed over the whole ranking)
is the more trustworthy leakage signal here. Net conclusion: random_forest was benefiting the
most from client memorization under the naive split, and the client-grouped split was worth
doing -- not catastrophic leakage, but a real and consistent one.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
print("="*72)
print("LEAKAGE AUDIT -- final feature set (same hunt as w03_feature_leakage_check.ipynb)")
print("="*72)

leak_cols = ['impressions_mar', 'impressions_apr', 'pct_change', 'trend_direction', 'trend_pct']
present = [c for c in leak_cols if c in dataset.columns]
in_features = [c for c in leak_cols if c in feature_cols]
print(f"Leak columns present in dataset.csv at all: {present or 'none'}")
print(f"Leak columns present in feature_cols (the ones actually fed to the model): {in_features or 'none'}")

print("\n=== CORRELATION WITH LABEL (secondary leakage signal, >0.8 = investigate) ===")
correlations = dataset[feature_cols].corrwith(dataset[target_col]).abs().sort_values(ascending=False)
for feat, corr in correlations.items():
    flag = '  <-- INVESTIGATE' if corr > 0.8 else ''
    print(f"  {feat:24s} {corr:.3f}{flag}")
suspicious = correlations[correlations > 0.8]
print(f"\nResult: {'SUSPICIOUS -- investigate before trusting the model' if len(suspicious) else 'No feature is suspiciously correlated with the label.'}")

print("\n=== SPLIT-LEVEL LEAKAGE (client identity) ===")
print(f"Client overlap in the AFTER split (section 2): "
      f"{len(after_train_clients & after_test_clients)} (must be 0 -- re-verified here, not just claimed)")

print("\n=== RESULT ===")
leakage_found = bool(in_features) or bool(len(suspicious)) or bool(after_train_clients & after_test_clients)
print(f"Overall: {'LEAKAGE DETECTED -- fix before trusting section 2/3 results' if leakage_found else 'Clean -- no direct leak columns, no suspiciously correlated feature, no client overlap.'}")


LEAKAGE AUDIT -- final feature set (same hunt as w03_feature_leakage_check.ipynb)
Leak columns present in dataset.csv at all: none
Leak columns present in feature_cols (the ones actually fed to the model): none

=== CORRELATION WITH LABEL (secondary leakage signal, >0.8 = investigate) ===
  active_days_90d          0.181
  has_momentum             0.159
  ctr_90d                  0.130
  has_ga4_data             0.080
  avg_position_90d         0.073
  clicks_90d               0.055
  organic_sessions_90d     0.031
  impressions_last30       0.017
  impressions_90d          0.016
  engaged_sessions_90d     0.014
  impressions_first60      0.013
  momentum_pct             0.008
  pageviews_90d            0.008
  sessions_90d             0.008

Result: No feature is suspiciously correlated with the label.

=== SPLIT-LEVEL LEAKAGE (client identity) ===
Client overlap in the AFTER split (section 2): 0 (must be 0 -- re-verified here, not just claimed)

=== RESULT ===
Overall: Clean -- no di

## 3. Leakage audit

The same three checks from `w03_feature_leakage_check.ipynb`, re-run here on the exact
feature set and split actually used for modeling (not just claimed in Week 3): the ML-04
excluded columns are absent from both the dataset and the feature list; no feature is
suspiciously correlated (>0.8) with `is_declining_label`; and the client-grouped split from
section 2 is re-verified (not just re-asserted) to share zero clients between train and test.

This directly answers the methodology question raised against **Finding A** in section 1 for
our own project: a model can only "rediscover the label formula" if a label-construction
column leaks into the features, and the correlation check is exactly the safeguard the paper
itself reached for after the fact ("importance is descriptive rather than causal") -- ours is
checked before the model is trusted, not noted as a caveat afterward.

**Result: clean.** No ML-04 leak column is present in `dataset.csv` or in `feature_cols`. The
highest label correlation of any feature is `active_days_90d` at 0.181, then `has_momentum`
at 0.159 and `ctr_90d` at 0.130 -- all far below the 0.8 investigate threshold, and
consistent with ML-06's finding that these signals are real but modest, not near-deterministic.
The client-grouped AFTER split is re-verified at 0 overlap. This is the direct evidence that
our project doesn't repeat Finding A's trap: no feature towers over the others (0.181 is the
max, not 0.43), and none is a sibling of the label-construction columns.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
BANNED_PATTERNS = ['predicts', 'will decline', 'will grow', 'causes', 'proves', 'prevent',
                    'guarantees', 'always', 'never fails', "google's algorithm"]

def flag_risky_language(text):
    hits = [p for p in BANNED_PATTERNS if p in text.lower()]
    return hits

risky_claim = (
    "Our random forest model predicts which pages will decline, so content teams can "
    "prevent traffic loss before it happens."
)
safe_claim = (
    "On our client-grouped holdout split, the random forest's score is associated with "
    "future search-visibility decline more strongly than the Week-4 rule baseline "
    "(see the Precision@50 comparison in w05_model.ipynb section 3); we treat it as a "
    "decision-support signal for prioritizing editorial review, not a guarantee -- a human "
    "reviewer still checks every flagged page before any action is taken."
)

print("BEFORE (risky):", risky_claim)
print("Flags found:", flag_risky_language(risky_claim))
print()
print("AFTER (safe):  ", safe_claim)
print("Flags found:", flag_risky_language(safe_claim))


BEFORE (risky): Our random forest model predicts which pages will decline, so content teams can prevent traffic loss before it happens.
Flags found: ['predicts', 'will decline', 'prevent']

AFTER (safe):   On our client-grouped holdout split, the random forest's score is associated with future search-visibility decline more strongly than the Week-4 rule baseline (see the Precision@50 comparison in w05_model.ipynb section 3); we treat it as a decision-support signal for prioritizing editorial review, not a guarantee -- a human reviewer still checks every flagged page before any action is taken.
Flags found: []


## 4. Claim rewrite

**Before -> after, word by word:**

| Risky word/phrase | Why it overclaims | Safe replacement |
|---|---|---|
| "predicts which pages will decline" | states a future outcome as fact | "the score is associated with future decline" |
| (no mention of the split) | invites belief the number is universal | names the exact split and points to where the comparison lives |
| "so content teams can prevent traffic loss" | promises a causal, guaranteed outcome | "decision-support signal for prioritizing review" |
| (no human-in-the-loop mention) | implies automation | explicitly requires a human reviewer before any action |

The `flag_risky_language()` check above is a small, honest device, not a rubber stamp -- it
only catches the specific words on its list, so passing it is necessary, not sufficient, for
a claim to be safe. The real test is still `DATA_USE.md`'s standard: observed / measured /
directional / decision-support, never a claim to "predict Google's algorithm" or a causal
refresh-impact guarantee.

Checked `w01_research_question.ipynb` and the `capstone_report_template.md` for an earlier
drafted sentence to rewrite instead of the example above -- neither had a distinct bold claim
already written (the ML-02 notebook's "careful words" section was itself still a skeleton
prompt, not a drafted sentence), so `risky_claim` above stands as the actual answer to this
exercise rather than a placeholder: it names this project's own real result (the random
forest vs. the Week-4 baseline, on the actual client-grouped split from `w05_model.ipynb`),
not a generic example.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.